# 02 — Chunking Demo

Full pipeline: **PDF → extract blocks → classify → chunk**

This notebook demonstrates Layers 1 → 2 → 4 of the HSC-Edu pipeline,
ending with semantic chunks ready for embedding.

In [1]:
from pathlib import Path

from hsc_edu.core.extraction import extract_document
from hsc_edu.core.classification import classify_blocks
from hsc_edu.core.chunking import chunk_blocks, count_tokens
from hsc_edu.core.models import BlockType

DATA_DIR = Path("../data")
SAMPLE_PDF = DATA_DIR / "Java.pdf"

print(f"PDF: {SAMPLE_PDF.name}  ({SAMPLE_PDF.stat().st_size / 1024 / 1024:.1f} MB)")

PDF: Java.pdf  (6.3 MB)


## 1. Extract blocks (Layer 1)

In [2]:
blocks = extract_document(SAMPLE_PDF, doc_id="java-demo")
print(f"Extracted {len(blocks)} blocks from '{SAMPLE_PDF.name}'")

Extracted 2121 blocks from 'Java.pdf'


## 2. Classify blocks (Layer 2)

In [3]:
classified = classify_blocks(blocks)

from collections import Counter

type_counts = Counter(b.block_type for b in classified)
print(f"Classified {len(classified)} blocks:")
for bt, cnt in type_counts.most_common():
    print(f"  {bt:<20s} {cnt:>5d}")

Classified 2121 blocks:
  paragraph             1981
  heading                134
  exercise                 5
  note                     1


In [4]:
headings = [b for b in classified if b.block_type == BlockType.HEADING]
print(f"Headings found: {len(headings)}\n")

for h in headings[:15]:
    indent = "  " * ((h.heading_level or 1) - 1)
    text = h.raw_text.split("\n")[0][:80]
    print(f"  p.{h.page:>3d}  lv{h.heading_level}  {indent}{text}")

Headings found: 134

  p. 11  lv2    1.1. KHÁI NIỆM CƠ BẢN
  p. 12  lv2    1.2. ĐỐI TƯỢNG VÀ LỚP
  p. 14  lv2    1.3. CÁC NGUYÊN TẮC TRỤ CỘT
  p. 19  lv2    2.1. ĐẶC TÍNH CỦA JAVA
  p. 20  lv3      2.1.1. Máy ảo Java – Java Virtual Machine
  p. 23  lv3      2.1.4. Cấu trúc mã nguồn Java
  p. 24  lv3      2.1.5. Chương trình Java đầu tiên
  p. 26  lv2    2.2. BIẾN
  p. 28  lv3      2.3.3. Các phép toán khác
  p. 30  lv3      2.4.1. Các cấu trúc rẽ nhánh
  p. 36  lv3      2.4.2. Các cấu trúc lặp
  p. 42  lv3      2.4.3. Biểu thức điều kiện trong các cấu trúc điều khiển
  p. 45  lv4        i) Sử dụng vòng lặp để in ra các số từ 1 đến 10 trên một dòng, dùng kí tự tab 
  p. 48  lv2    3.1. TẠO VÀ SỬ DỤNG ĐỐI TƯỢNG
  p. 50  lv2    3.2. TƯƠNG TÁC GIỮA CÁC ĐỐI TƯỢNG


## 3. Chunk (Layer 4)

In [5]:
chunks = chunk_blocks(classified)

print(f"Total chunks: {len(chunks)}")
print(f"Token range : {min(c.token_count for c in chunks)} – {max(c.token_count for c in chunks)}")
print(f"Avg tokens  : {sum(c.token_count for c in chunks) / len(chunks):.0f}")

Total chunks: 264
Token range : 12 – 1067
Avg tokens  : 680


### 3.1 Token distribution

In [6]:
buckets = {"0-64": 0, "64-256": 0, "256-512": 0, "512-1024": 0, ">1024": 0}
for c in chunks:
    t = c.token_count
    if t <= 64:
        buckets["0-64"] += 1
    elif t <= 256:
        buckets["64-256"] += 1
    elif t <= 512:
        buckets["256-512"] += 1
    elif t <= 1024:
        buckets["512-1024"] += 1
    else:
        buckets[">1024"] += 1

print("Token distribution:")
for label, cnt in buckets.items():
    bar = "█" * cnt
    print(f"  {label:>10s}: {cnt:>4d}  {bar}")

Token distribution:
        0-64:   41  █████████████████████████████████████████
      64-256:    5  █████
     256-512:   29  █████████████████████████████
    512-1024:  178  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
       >1024:   11  ███████████


## 4. First 10 chunks (with section path)

Each chunk shows its **heading hierarchy** (`section_path`) so you can see the table-of-contents context.

In [7]:
SHOW_N = 10

for i, ch in enumerate(chunks[:SHOW_N]):
    path_str = " > ".join(ch.section_path) if ch.section_path else "(no heading)"
    text_preview = ch.text[:200].replace("\n", " ↵ ")
    if len(ch.text) > 200:
        text_preview += " …"

    print(f"{'═' * 72}")
    print(f"Chunk {i}  │  {ch.token_count} tokens  │  pages {ch.page_start}–{ch.page_end}  │  type: {ch.block_type}")
    print(f"Section: {path_str}")
    print(f"Text   : {text_preview}")
    print()

════════════════════════════════════════════════════════════════════════
Chunk 0  │  1067 tokens  │  pages 0–1  │  type: paragraph
Section: (no heading)
Text   : Mục lục ↵  ↵ GIỚI THIỆU .............................................................................5 ↵  ↵ Chương 1. MỞ ĐẦU ............................................................7 ↵  ↵ 1.1. KHÁI NIỆM CƠ BẢ …

════════════════════════════════════════════════════════════════════════
Chunk 1  │  1038 tokens  │  pages 1–2  │  type: paragraph
Section: (no heading)
Text   : 5.6. BIẾN THỰC THỂ VÀ BIẾN ĐỊA PHƯƠNG ........... 80 ↵  ↵ Chương 6. SỬ DỤNG THƯ VIỆN JAVA ......................... 85 ↵  ↵ 6.1. ArrayList ..................................................................... …

════════════════════════════════════════════════════════════════════════
Chunk 2  │  1031 tokens  │  pages 2–3  │  type: paragraph
Section: (no heading)
Text   : 9.3.1. Gọi hàm khởi tạo của lớp cha ........................ 150 ↵  ↵ 9.3.2. Truyền đố

## 5. Inspect a specific chunk

Change `CHUNK_IDX` to drill into any chunk.

In [17]:
CHUNK_IDX = 243  # Total 264

ch = chunks[CHUNK_IDX]

print(f"chunk_id     : {ch.chunk_id}")
print(f"doc_id       : {ch.doc_id}")
print(f"block_type   : {ch.block_type}")
print(f"token_count  : {ch.token_count}")
print(f"pages        : {ch.page_start} – {ch.page_end}")
print(f"chapter      : {ch.chapter}")
print(f"section      : {ch.section}")
print(f"section_path : {ch.section_path}")
print(f"block_ids    : {ch.block_ids}")
print(f"\n{'─' * 72}")
print(ch.text)

chunk_id     : 62ff9e8407b7
doc_id       : java-demo
block_type   : paragraph
token_count  : 1007
pages        : 216 – 217
chapter      : 13.1. LỚP TỔNG QUÁT
section      : 13.1. LỚP TỔNG QUÁT
section_path : ['13.1. LỚP TỔNG QUÁT']
block_ids    : ['ff7b48240e3d', '8d8c8802193e', '0fc754d149bf', 'e88dac0d8577', '3f4f008dd93f', 'dddb52a7dc02', '726a2bdaf23c', '01429b7aa851', 'ff6ac776f52f', 'a3f4bc2ea9f3', '617cd8a8f60f']

────────────────────────────────────────────────────────────────────────
13.1. LỚP TỔNG QUÁT

Lớp tổng quát là lớp mà trong khai báo có ít nhất một tham số kiểu. Lớp 
ArrayList mà ta đã gặp ở các chương trước là một ví dụ về lớp tổng quát trong thư 
viện chuẩn của Java. Một đối tượng ArrayList về bản chất là một mảng động chứa 
các tham chiếu kiểu Object. Do lớp nào cũng là lớp con của Object nên ArrayList có 
thể lưu trữ mọi thứ. Không chỉ vậy, ArrayList còn sử dụng một khái niệm của Java 
là "tham số kiểu", như ở ArrayList<String>, để giới hạn các giá trị có thể được